# ML Wizards – Group 5
## Track 5: Reproducibility / Benchmarking
### Fair Benchmark of XGBoost, Random Forest and MLP on Two Credit Datasets

**Goal:** compare three common classifiers on two public credit-risk datasets using the same train/test split for each dataset and random seed. We report Accuracy, Precision, Recall and F1-score. This is a class project, not a credit decision tool.

## 1. Setup

This first cell installs the required libraries in Google Colab. It may take a few minutes the first time. The models use CPU-friendly settings.

In [ ]:
%pip -q install ucimlrepo xgboost

## 2. Imports and experiment settings

We use three fixed seeds: 42, 21 and 7. For each dataset and seed, all three models receive exactly the same training and test rows. Stratification keeps the class proportions similar in both splits.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

SEEDS = [42, 21, 7]
TEST_SIZE = 0.20
METRICS = ["Accuracy", "Precision", "Recall", "F1-score"]
print("Seeds:", SEEDS, " | Test fraction:", TEST_SIZE)

## 3. Load the datasets

The data are downloaded from the UCI Machine Learning Repository. We remove the client ID from the credit-card data because it is only an identifier. For both datasets, the positive class means the risk/default outcome: German **bad credit** (class 2) and credit-card **default next month** (class 1).

In [ ]:
# UCI dataset IDs: Statlog German Credit = 144; Default of Credit Card Clients = 350
german = fetch_ucirepo(id=144)
card = fetch_ucirepo(id=350)

def combine_uci(dataset, target_name):
    X = dataset.data.features.copy()
    y = dataset.data.targets.iloc[:, 0].copy()
    # Remove ID if the source includes it; IDs do not describe credit risk.
    id_columns = [c for c in X.columns if str(c).strip().lower() in {"id", "id​"}]
    X = X.drop(columns=id_columns)
    return X, y

X_german, y_german = combine_uci(german, "class")
X_card, y_card = combine_uci(card, "default payment next month")

# Normalize labels to 0=lower risk / no default, 1=bad credit / default.
def map_binary_target(y, positive_value):
    values = pd.Series(y).dropna().unique()
    if len(values) != 2:
        raise ValueError(f"Expected two target classes, found {values}")
    return (pd.Series(y).astype(str) == str(positive_value)).astype(int)

y_german = map_binary_target(y_german, 2)
y_card = map_binary_target(y_card, 1)

datasets = {
    "German Credit (Statlog)": (X_german, y_german),
    "Default of Credit Card Clients": (X_card, y_card),
}
for name, (X, y) in datasets.items():
    print(f"{name}: {X.shape[0]} rows, {X.shape[1]} features; positive rate={y.mean():.3f}")
    display(X.head(3))

## 4. Preprocessing and model definitions

The preprocessing is part of each model pipeline. Numerical missing values are filled with the training median, then scaled. Categorical missing values are filled with the most common training value, then one-hot encoded. Integer columns with at most 10 distinct values are treated as category codes, a practical heuristic for these UCI datasets. The pipeline is fitted only on training data, preventing information from the test set leaking into preprocessing.

In [ ]:
def make_preprocessor(X):
    # UCI sometimes stores category codes as numbers. Treat integer columns
    # with few distinct values as categorical too (a practical heuristic).
    categorical_cols = []
    for col in X.columns:
        series = X[col]
        low_cardinality_code = (
            pd.api.types.is_integer_dtype(series)
            and series.nunique(dropna=True) <= 10
        )
        if (not pd.api.types.is_numeric_dtype(series)) or low_cardinality_code:
            categorical_cols.append(col)
    numeric_cols = [c for c in X.columns if c not in categorical_cols]
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_cols),
        ("categorical", categorical_pipe, categorical_cols),
    ], remainder="drop")

def make_models(seed):
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=100, min_samples_leaf=2, n_jobs=-1,
            random_state=seed),
        "XGBoost": XGBClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            eval_metric="logloss", n_jobs=-1, random_state=seed),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(32, 16), max_iter=100,
            early_stopping=True, n_iter_no_change=10,
            random_state=seed),
    }

print("Preprocessing and models are ready.")

## 5. Run the benchmark

For each dataset and seed, we create one stratified split, then reuse those same rows for every model. The loop records all four metrics. It also saves the seed-42 predictions for later confusion-matrix plots. Model fitting can take a few minutes.

In [ ]:
results = []
seed42_predictions = {}

for dataset_name, (X, y) in datasets.items():
    for seed in SEEDS:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=seed, stratify=y
        )
        for model_name, classifier in make_models(seed).items():
            pipeline = Pipeline([
                ("preprocess", make_preprocessor(X_train)),
                ("model", classifier),
            ])
            pipeline.fit(X_train, y_train)
            pred = pipeline.predict(X_test)
            results.append({
                "Dataset": dataset_name,
                "Seed": seed,
                "Model": model_name,
                "Accuracy": accuracy_score(y_test, pred),
                "Precision": precision_score(y_test, pred, zero_division=0),
                "Recall": recall_score(y_test, pred, zero_division=0),
                "F1-score": f1_score(y_test, pred, zero_division=0),
            })
            if seed == 42:
                seed42_predictions[(dataset_name, model_name)] = (y_test.to_numpy(), pred)
        print(f"Finished {dataset_name}, seed {seed}")

results_df = pd.DataFrame(results)
print("Runs completed:", len(results_df), "(expected 18)")
display(results_df.round(4))

## 6. Results tables and averages

The first table contains every individual run. This summary averages each metric across the three seeds for each dataset and model. These are measured results from the runs above, not example numbers.

In [ ]:
average_scores = (results_df.groupby(["Dataset", "Model"])[METRICS]
                 .mean().reset_index())
display(average_scores.round(4))

# Easy-to-read comparison table: average F1-score by dataset and model.
f1_table = average_scores.pivot(index="Dataset", columns="Model", values="F1-score")
display(f1_table.round(4))

## 7. Charts

These charts compare the mean score over the three seeds. Higher values are better for all four metrics. The plotted values are calculated directly from the results table.

In [ ]:
for metric in METRICS:
    ax = (average_scores.pivot(index="Dataset", columns="Model", values=metric)
          .plot(kind="bar", figsize=(9, 4), ylim=(0, 1), rot=12,
                title=f"Mean {metric} across {len(SEEDS)} seeds"))
    ax.set_ylabel(metric)
    ax.set_xlabel("")
    ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## 8. Confusion matrices

A confusion matrix shows correct and incorrect predictions as counts. To keep the display compact, these matrices use the **seed 42 test split** for each model and dataset. Rows are the true class; columns are the predicted class.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for row, dataset_name in enumerate(datasets.keys()):
    for col, model_name in enumerate(["Random Forest", "XGBoost", "MLP"]):
        y_true, y_pred = seed42_predictions[(dataset_name, model_name)]
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        disp = ConfusionMatrixDisplay(cm, display_labels=["No risk/default", "Bad credit/default"])
        disp.plot(ax=axes[row, col], colorbar=False, values_format="d")
        axes[row, col].set_title(f"{dataset_name}\n{model_name}")
plt.tight_layout()
plt.show()

## 9. Limitations

- These two historical datasets are small or specific to their original populations and time periods; results may not transfer to current lending.
- A single 80/20 split per seed gives a useful repeatable comparison, but it is not as thorough as nested cross-validation or a larger repeated evaluation.
- The models use simple fixed settings rather than extensive hyperparameter tuning. The goal is a fair, quick baseline comparison.
- Accuracy can hide poor performance on the less common positive class, so Precision, Recall and F1-score are also reported.
- Credit decisions have real consequences. This educational benchmark does not assess fairness across protected groups, explainability, calibration, legal compliance, or suitability for real decisions.

## 10. Conclusion

Use the generated mean-score table and charts to state which model performed best on each dataset and metric. Discuss whether that result was consistent across all three seeds. Do not assume one model is universally best: the two datasets differ, and these runs are a limited benchmark. This section is intentionally a prompt for conclusions based on the results produced when you run the notebook.

## 11. References

1. UCI Machine Learning Repository, [Statlog (German Credit Data), dataset 144](https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data).
2. UCI Machine Learning Repository, [Default of Credit Card Clients, dataset 350](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients).
3. Chen, T. & Guestrin, C. (2016). [XGBoost: A Scalable Tree Boosting System](https://doi.org/10.1145/2939672.2939785).
4. scikit-learn documentation: [Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html), [classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics), and [MLPClassifier](https://scikit-learn.org/stable/modules/neural_networks_supervised.html).

**Reproducibility note:** the notebook records fixed split and model seeds. Package versions and downloaded dataset revisions can change, so for formal reproducibility also record the package versions and date you ran this notebook.